In [1]:
import pandas as pd
import csv
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.stats import f_oneway
from matplotlib.backends.backend_pdf import PdfPages

In [2]:
def cluster_analysis_and_weather_comparison(file_path, n_clusters=3):
    print(file_path.split('/')[-1].split('.')[0].split('_')[3:][0] + " " +
         '(' + file_path.split('/')[-1].split('.')[0].split('_')[3:][1] + 
          file_path.split('/')[-1].split('.')[0].split('_')[3:][2] +
         ')')
    os.environ["OMP_NUM_THREADS"] = "1"
    sns.set_style("darkgrid")  # Use fancy seaborn style for better visuals
    
    df = pd.read_csv(file_path)
    nutrient_features = ["NPK", "PK", "NK", "CK", "OF"]
    df = df.dropna(subset=nutrient_features)
    
    scaler = StandardScaler()
    df_scaled = scaler.fit_transform(df[nutrient_features])
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df["Cluster"] = kmeans.fit_predict(df_scaled)
    df.to_csv("clustered_water_sample.csv", index=False)
    
    print("\nCluster Counts:")
    print(df["Cluster"].value_counts())
    
    # Perform PCA for visualization
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(df_scaled)
    df["PCA1"] = pca_result[:, 0]
    df["PCA2"] = pca_result[:, 1]
    
    # Scatter Plot for Clusters
    cluster_colors = {0: "red", 1: "green", 2: "blue"}
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        x="PCA1", y="PCA2", hue="Cluster", palette=cluster_colors, 
        data=df, alpha=0.7, s=150, edgecolor="black"
    )
    plt.xlabel("Principal Component 1", fontsize=14)  # Increased font size for x label
    plt.ylabel("Principal Component 2", fontsize=14)  # Increased font size for y label
    plt.title("K-Means Clustering (PCA Reduced)" + " " + "-- " +
          file_path.split('/')[-1].split('.')[0].split('_')[3:][0] + " " +
         '(' + file_path.split('/')[-1].split('.')[0].split('_')[3:][1] + " " +
          file_path.split('/')[-1].split('.')[0].split('_')[3:][2] + 
         ')', fontsize=14, fontweight='bold')
    
    plt.legend(title="Cluster", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.7)
    
    # Add a thinner box border to the plot
    ax = plt.gca()
    for _, spine in ax.spines.items():
        spine.set_edgecolor('black')
        spine.set_linewidth(0.8)  # Thinner border
    
    # Save scatter plot to the first folder
    os.makedirs(prefex + 'scatter_plots', exist_ok=True)
    plt.savefig(prefex + 'scatter_plots/' + file_path.split('/')[-1].split('.')[0] 
                + 'cluster_scatter_plot.png', bbox_inches='tight')
    plt.close()
    
    # Weather Feature Analysis
    weather_features = [
        "Dew Point Temperature (F)", "Visibility (mi)", "Average Wind Speed (knots)",
        "Maximum Sustained Wind Speed (knots)", "Maximum Gust (knots)", "Maximum Temperature (F)",
        "Minimum Temperature (F)", "Precipitation (in)"
    ]
    
    cluster_summary = df.groupby("Cluster")[weather_features].mean()
    print("\nWeather Condition Summary by Cluster:")
    print(cluster_summary)
    
    # Bar Plot for Weather Summary
    plt.figure(figsize=(16, 8))
    ax = cluster_summary.T.plot(
        kind="bar", figsize=(14, 7), color=["red", "green", "blue"], edgecolor="black", linewidth=0.8
    )
    plt.xlabel("Weather Variables", fontsize=14)  # Increased font size for x label
    plt.ylabel("Average Value", fontsize=14)  # Increased font size for y label
    plt.title("Weather Conditions Across Clusters" + " " + "-- " +
          file_path.split('/')[-1].split('.')[0].split('_')[3:][0] + " " +
         '(' + file_path.split('/')[-1].split('.')[0].split('_')[3:][1] + " " +
          file_path.split('/')[-1].split('.')[0].split('_')[3:][2] + 
         ')', fontsize=14, fontweight='bold')
    
    plt.xticks(rotation=90, fontsize=14)
    plt.yticks(fontsize=14)
    plt.legend(title="Cluster", fontsize=14)
    plt.grid(axis="y", linestyle="--", alpha=0.7)  # Dashed grid for better readability
    plt.axhline(y=0, color='black', linewidth=1)
    # Add a thinner box border to the plot
    for _, spine in ax.spines.items():
        spine.set_edgecolor('black')
        spine.set_linewidth(0.8)  # Thinner border
    
    # Save bar plot to the second folder
    os.makedirs(prefex + 'weather_Conditions_Across_Clusters', exist_ok=True)
    plt.savefig(prefex + 'weather_Conditions_Across_Clusters/' 
                + file_path.split('/')[-1].split('.')[0] 
                + 'weather_bar_plot.png', bbox_inches='tight')
    plt.close()
    
    # ANOVA Analysis for Weather Features
    print("\nStatistical Comparison (ANOVA p-values):")
    for feature in weather_features:
        groups = [df[df["Cluster"] == c][feature].dropna() for c in df["Cluster"].unique()]
        stat, p = f_oneway(*groups)
        print(f"{feature}: p-value = {p:.4f}")
    
    return df


In [3]:
#prefex = '/Users/rjing/Desktop/Machine_Learning_Nonpoint_Source_Pollution/data/water_runoff_sample/Total_Phosphorus/'
#prefex = '/Users/rjing/Desktop/Machine_Learning_Nonpoint_Source_Pollution/data/water_runoff_sample/Total_Nitrogen/'
#prefex = '/Users/rjing/Desktop/Machine_Learning_Nonpoint_Source_Pollution/data/water_runoff_sample/Ammoniacal_Nitrogen/'
#prefex = '/Users/rjing/Desktop/Machine_Learning_Nonpoint_Source_Pollution/data/water_runoff_sample/Dissolved_Phosphorus/'
#prefex = '/Users/rjing/Desktop/Machine_Learning_Nonpoint_Source_Pollution/data/water_runoff_sample/Nitrate_Nitrogen/'
prefex = '/Users/rjing/Desktop/Machine_Learning_Nonpoint_Source_Pollution/data/water_runoff_sample/Particulate_Phosphorus/'

file_list = [f for f in os.listdir(prefex) if f.endswith(".csv")]
print(file_list)

selected_files = []
# Filter files that match the "Total_Nitrogen" condition
for file in file_list:
    df_clustered = cluster_analysis_and_weather_comparison(prefex + file,  n_clusters=3)


print("Job done!")

['Water_sample_runoff_citrus_Particulate_Phosphorus.csv', 'Water_sample_runoff_corn_Particulate_Phosphorus.csv', 'Water_sample_runoff_rice_Particulate_Phosphorus.csv', 'Water_sample_runoff_vegetable_Particulate_Phosphorus.csv']
citrus (ParticulatePhosphorus)

Cluster Counts:
Cluster
0    32
1    24
2    16
Name: count, dtype: int64


C:\Users\rjing\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



Weather Condition Summary by Cluster:
         Dew Point Temperature (F)  Visibility (mi)  \
Cluster                                               
0                        -0.028774         0.007027   
1                        -0.056920         0.108218   
2                         0.208885         0.560599   

         Average Wind Speed (knots)  Maximum Sustained Wind Speed (knots)  \
Cluster                                                                     
0                          0.433458                              0.198976   
1                          0.409277                              0.304358   
2                          0.107624                              0.032056   

         Maximum Gust (knots)  Maximum Temperature (F)  \
Cluster                                                  
0                   -0.217312                -0.138092   
1                    0.681527                -0.098827   
2                    0.345622                 0.333938   

        

C:\Users\rjing\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



Weather Condition Summary by Cluster:
         Dew Point Temperature (F)  Visibility (mi)  \
Cluster                                               
0                         0.278617         0.265361   
1                         0.352619        -0.057446   
2                         0.476769        -0.047565   

         Average Wind Speed (knots)  Maximum Sustained Wind Speed (knots)  \
Cluster                                                                     
0                          0.130576                              0.078721   
1                          0.123155                              0.222740   
2                         -0.260542                             -0.068382   

         Maximum Gust (knots)  Maximum Temperature (F)  \
Cluster                                                  
0                   -0.657766                 0.502540   
1                    0.519064                 0.283294   
2                   -0.335340                 0.356402   

        

...


Weather Condition Summary by Cluster:
         Dew Point Temperature (F)  Visibility (mi)  \
Cluster                                               
0                        -0.275501        -0.285932   
1                        -0.403843         0.260599   
2                        -0.964233         0.132027   

         Average Wind Speed (knots)  Maximum Sustained Wind Speed (knots)  \
Cluster                                                                     
0                          0.170016                              0.019528   
1                          0.212547                              0.132054   
2                          3.117591                              2.027404   

         Maximum Gust (knots)  Maximum Temperature (F)  \
Cluster                                                  
0                   -0.049471                -0.315996   
1                    0.177162                -0.299788   
2                   -0.896917                -0.472742   

        

<Figure size 1600x800 with 0 Axes>

<Figure size 1600x800 with 0 Axes>

<Figure size 1600x800 with 0 Axes>

<Figure size 1600x800 with 0 Axes>